# CPE 342 Term Project — Master Submission Merger & Quality Gate
**Course:** CPE 342 Machine Learning (Semester 1/2026) — KMUTT  
**Purpose:** รวมผลทำนายจากทั้ง 5 Tasks (`task1`–`task5/predictions/test_predictions.csv`) เข้าสู่ไฟล์ `submission.csv` (`25,889` แถว $\times 6$ คอลัมน์: `id,task1,task2,task3,task4,task5`) พร้อมตรวจสอบความถูกต้องตามกฎของ `project/AGENTS.md` และ Kaggle Leaderboard 100%

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR / "project" if (BASE_DIR / "project").exists() else BASE_DIR

sample_sub = pd.read_csv(PROJECT_DIR / "sample_submission.csv")
print(f"Loaded sample_submission.csv: {sample_sub.shape}")
display(sample_sub.head(5))

In [ ]:
sub = pd.DataFrame({"id": sample_sub["id"]})

for t_idx in range(1, 6):
    col = f"task{t_idx}"
    pred_path = PROJECT_DIR / f"task{t_idx}" / "predictions" / "test_predictions.csv"
    if pred_path.exists():
        df_p = pd.read_csv(pred_path)
        assert len(df_p) == 25889, f"{col} row count mismatch: {len(df_p)} != 25889"
        sub[col] = df_p[col].values
        print(f"✓ Loaded {col} from {pred_path}")
    else:
        print(f"⚠ {pred_path} not generated yet — filling placeholder 0 for {col}")
        sub[col] = 0

# Enforce strict dtypes & value ranges
sub["task1"] = sub["task1"].fillna(0).astype(int)
sub["task2"] = sub["task2"].fillna(0).astype(int)
sub["task3"] = np.round(np.clip(sub["task3"].fillna(0.0).astype(float), 0.0, None), 2)
if sub["task4"].dtype != object:
    sub["task4"] = sub["task4"].fillna(0).astype(int)
sub["task5"] = sub["task5"].fillna(0).astype(int)

# Strict Quality Gate Assertions
assert sub.shape == (25889, 6), f"Invalid shape: {sub.shape}"
assert list(sub.columns) == ["id", "task1", "task2", "task3", "task4", "task5"], f"Invalid columns: {list(sub.columns)}"
assert sub.isna().sum().sum() == 0, "Submission contains NaN values!"
assert (sub["id"].values == sample_sub["id"].values).all(), "ID order mismatch!"
assert set(sub["task1"].unique()).issubset({0, 1}), f"Invalid task1 values: {sub['task1'].unique()}"
assert set(sub["task2"].unique()).issubset({0, 1, 2, 3}), f"Invalid task2 values: {sub['task2'].unique()}"
assert (sub["task3"] >= 0).all(), "Negative values found in task3!"
assert set(sub["task5"].unique()).issubset({0, 1}), f"Invalid task5 values: {sub['task5'].unique()}"

out_csv = PROJECT_DIR / "submission.csv"
sub.to_csv(out_csv, index=False)
print(f"\n✅ Verified and saved final Kaggle submission: {out_csv} ({sub.shape})")
display(sub.head(10))